In [1]:
## Part 1: Set Up Environment and Data (10 min)
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random

# Stop the existing Spark session if it exists
if 'spark' in globals() and spark.sparkContext._jsc is not None:
    spark.stop()
    print("Spark session stopped.")

# Create a new Spark session
spark = SparkSession.builder \
    .appName("StreamPulse-PlanAudit") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

In [2]:
# Generate the StreamPulse dataset:
random.seed(42)

# Listening events (large table - 500K rows)
events_data = []
for i in range(500000):
    events_data.append((
        f"EVT-{i+1:07d}",
        f"USR-{random.randint(1, 100000):06d}",
        f"TRK-{random.randint(1, 50000):06d}",
        f"ART-{random.randint(1, 5000):05d}",
        random.randint(10, 300),
        random.choice([True, False]),
        random.choice(["mobile", "desktop", "smart_speaker", "tablet"]),
        random.choice(["free", "premium"]),
        f"202{random.randint(3,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))

events = spark.createDataFrame(events_data,
    ["event_id", "user_id", "track_id", "artist_id", "duration_sec",
     "completed", "device", "tier", "event_date"]) \
    .withColumn("event_date", col("event_date").cast("date")) \
    .withColumn("year", year(col("event_date"))) \
    .withColumn("month", month(col("event_date")))

events.write.parquet("audit_data/events", mode="overwrite", partitionBy=["year"])

# Artists (small table - 5K rows)
artist_data = [(f"ART-{i+1:05d}", f"Artist {i+1}",
                random.choice(["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic"]),
                random.choice(["US", "UK", "KR", "JP", "DE"]))
               for i in range(5000)]
artists = spark.createDataFrame(artist_data, ["artist_id", "name", "genre", "country"])
artists.write.parquet("audit_data/artists", mode="overwrite")

# Tracks (medium table - 50K rows)
track_data = [(f"TRK-{i+1:06d}", f"Track {i+1}",
               f"ART-{random.randint(1, 5000):05d}",
               random.randint(60, 400),
               random.randint(2018, 2024))
              for i in range(50000)]
tracks = spark.createDataFrame(track_data,
    ["track_id", "title", "artist_id", "track_duration", "release_year"])
tracks.write.parquet("audit_data/tracks", mode="overwrite")

# Reload from Parquet
events = spark.read.parquet("audit_data/events")
artists = spark.read.parquet("audit_data/artists")
tracks = spark.read.parquet("audit_data/tracks")

print(f"Events: {events.count()} | Artists: {artists.count()} | Tracks: {tracks.count()}")


Events: 500000 | Artists: 5000 | Tracks: 50000


In [3]:
## Query 1: Simple filter and select
q1 = events.filter(col("year") == 2024) \
    .filter(col("completed") == True) \
    .select("event_id", "user_id", "duration_sec")

print("QUERY 1: Simple filter and select")
q1.explain(mode="formatted")


QUERY 1: Simple filter and select
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [event_id#21, user_id#22, duration_sec#25L, completed#26, year#31]
Batched: true
Location: InMemoryFileIndex [file:/content/audit_data/events]
PartitionFilters: [isnotnull(year#31), (year#31 = 2024)]
PushedFilters: [IsNotNull(completed), EqualTo(completed,true)]
ReadSchema: struct<event_id:string,user_id:string,duration_sec:bigint,completed:boolean>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [event_id#21, user_id#22, duration_sec#25L, completed#26, year#31]

(3) Filter [codegen id : 1]
Input [5]: [event_id#21, user_id#22, duration_sec#25L, completed#26, year#31]
Condition : (isnotnull(completed#26) AND completed#26)

(4) Project [codegen id : 1]
Output [3]: [event_id#21, user_id#22, duration_sec#25L]
Input [5]: [event_id#21, user_id#22, duration_sec#25L, completed#26, year#31]




In [ ]:
##
Aspect               | Finding
--------------------|---------------------------------------------------
Scan type           | FileScan parquet
PartitionFilters    | [year#31 = 2024] (partition pruning applied)
PushedFilters       | [IsNotNull(completed), EqualTo(completed,true)]
ReadSchema columns  | event_id, user_id, duration_sec, year, completed
Exchange count      | 0
Assessment          | Efficient

In [4]:
## Query 2: Join events with artists
q2 = events.join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .select("event_id", "name", "genre", "duration_sec")

print("QUERY 2: Events JOIN Artists (filter after join)")
q2.explain(mode="formatted")


QUERY 2: Events JOIN Artists (filter after join)
== Physical Plan ==
* Project (13)
+- * SortMergeJoin Inner (12)
   :- * Sort (6)
   :  +- Exchange (5)
   :     +- * Project (4)
   :        +- * Filter (3)
   :           +- * ColumnarToRow (2)
   :              +- Scan parquet  (1)
   +- * Sort (11)
      +- Exchange (10)
         +- * Filter (9)
            +- * ColumnarToRow (8)
               +- Scan parquet  (7)


(1) Scan parquet 
Output [4]: [event_id#21, artist_id#24, duration_sec#25L, year#31]
Batched: true
Location: InMemoryFileIndex [file:/content/audit_data/events]
PartitionFilters: [isnotnull(year#31), (year#31 = 2024)]
PushedFilters: [IsNotNull(artist_id)]
ReadSchema: struct<event_id:string,artist_id:string,duration_sec:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [event_id#21, artist_id#24, duration_sec#25L, year#31]

(3) Filter [codegen id : 1]
Input [4]: [event_id#21, artist_id#24, duration_sec#25L, year#31]
Condition : isnotnull(artist_id#24)

(4) Project [c

In [ ]:
Aspect                   | Finding
-------------------------|---------------------------------------------------
Join strategy            | BroadcastHashJoin (BuildRight)
Artists table size       | ~5K rows - small
Exchange count           | 1 (BroadcastExchange)
Could broadcast?         | Already broadcasting ✓
Filter placement         | Filter applied AFTER join (processes all data)
Assessment               | Needs Optimization

In [5]:
## Query 3: Three-table join
q3 = events.join(tracks, "track_id") \
    .join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .filter(col("genre") == "Pop") \
    .groupBy("name") \
    .agg(count("*").alias("play_count"), avg("duration_sec").alias("avg_duration"))

print("QUERY 3: Three-table join with aggregation")
q3.explain(mode="formatted")


QUERY 3: Three-table join with aggregation
== Physical Plan ==
* HashAggregate (26)
+- Exchange (25)
   +- * HashAggregate (24)
      +- * Project (23)
         +- * SortMergeJoin Inner (22)
            :- * Sort (15)
            :  +- Exchange (14)
            :     +- * Project (13)
            :        +- * SortMergeJoin Inner (12)
            :           :- * Sort (6)
            :           :  +- Exchange (5)
            :           :     +- * Project (4)
            :           :        +- * Filter (3)
            :           :           +- * ColumnarToRow (2)
            :           :              +- Scan parquet  (1)
            :           +- * Sort (11)
            :              +- Exchange (10)
            :                 +- * Filter (9)
            :                    +- * ColumnarToRow (8)
            :                       +- Scan parquet  (7)
            +- * Sort (21)
               +- Exchange (20)
                  +- * Project (19)
                     +- * Filt

In [ ]:
Aspect                      | Finding
----------------------------|---------------------------------------------------
Join 1 strategy (events-tracks) | SortMergeJoin
Join 2 strategy (with artists) | BroadcastHashJoin
Total Exchange count        | 4 (3 joins + 1 aggregation)
Filter on year?             | Yes - applied before first join (pushed to scan)
Filter on genre?            | Applied AFTER joins (not pushed to artists scan)
Assessment                  | Needs Optimization

In [6]:
## Query 4a: Genre aggregation
enriched = events.join(artists, "artist_id").filter(col("year") == 2024)

print("QUERY 4a: Genre aggregation")
q4a = enriched.groupBy("genre").agg(count("*").alias("plays"))
q4a.explain(mode="formatted")

print("\nQUERY 4b: Device aggregation (same enriched source)")
q4b = enriched.groupBy("device").agg(avg("duration_sec").alias("avg_dur"))
q4b.explain(mode="formatted")


QUERY 4a: Genre aggregation
== Physical Plan ==
* HashAggregate (16)
+- Exchange (15)
   +- * HashAggregate (14)
      +- * Project (13)
         +- * SortMergeJoin Inner (12)
            :- * Sort (6)
            :  +- Exchange (5)
            :     +- * Project (4)
            :        +- * Filter (3)
            :           +- * ColumnarToRow (2)
            :              +- Scan parquet  (1)
            +- * Sort (11)
               +- Exchange (10)
                  +- * Filter (9)
                     +- * ColumnarToRow (8)
                        +- Scan parquet  (7)


(1) Scan parquet 
Output [2]: [artist_id#24, year#31]
Batched: true
Location: InMemoryFileIndex [file:/content/audit_data/events]
PartitionFilters: [isnotnull(year#31), (year#31 = 2024)]
PushedFilters: [IsNotNull(artist_id)]
ReadSchema: struct<artist_id:string>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [artist_id#24, year#31]

(3) Filter [codegen id : 1]
Input [2]: [artist_id#24, year#31]
Condition : isnotnu

In [ ]:
Aspect                      | Finding
----------------------------|---------------------------------------------------
Does 4a and 4b share computation? | NO - both execute join independently
Is enriched cached?         | NO
Redundant work              | Complete recomputation of events-artists join ×2
Assessment                  | Critical Issue

In [7]:
## Query 5: Self-join pattern
popular = events.groupBy("track_id").agg(count("*").alias("play_count")) \
    .filter(col("play_count") > 10)

q5 = events.join(popular, "track_id") \
    .select("event_id", "user_id", "track_id", "play_count")

print("QUERY 5: Self-reference (events aggregated then joined back)")
q5.explain(mode="formatted")


QUERY 5: Self-reference (events aggregated then joined back)
== Physical Plan ==
* Project (17)
+- * SortMergeJoin Inner (16)
   :- * Sort (6)
   :  +- Exchange (5)
   :     +- * Project (4)
   :        +- * Filter (3)
   :           +- * ColumnarToRow (2)
   :              +- Scan parquet  (1)
   +- * Sort (15)
      +- * Filter (14)
         +- * HashAggregate (13)
            +- Exchange (12)
               +- * HashAggregate (11)
                  +- * Project (10)
                     +- * Filter (9)
                        +- * ColumnarToRow (8)
                           +- Scan parquet  (7)


(1) Scan parquet 
Output [4]: [event_id#21, user_id#22, track_id#23, year#31]
Batched: true
Location: InMemoryFileIndex [file:/content/audit_data/events]
PushedFilters: [IsNotNull(track_id)]
ReadSchema: struct<event_id:string,user_id:string,track_id:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [event_id#21, user_id#22, track_id#23, year#31]

(3) Filter [codegen id : 1]
Input [4]:

In [ ]:
Aspect                      | Finding
----------------------------|---------------------------------------------------
How many times is events scanned? | 2 times
Exchange count              | 2 (1 aggregation shuffle + 1 BroadcastExchange)
Join strategy               | BroadcastHashJoin
Could caching help?         | Yes - cache popular aggregation result
Assessment                  | Needs Optimization

In [ ]:
## Part 3: Plan Analysis Report (20 min)
# Write a structured findings report:
Query 1: Simple Filter
Status: Efficient

Issues found: None - optimal execution with partition pruning, filter pushdown, and no shuffles

Recommendation: No changes needed - this query is well-optimized
###############
Query 2: Events-Artists Join
Status: Needs Optimization

Issues found:

Filter on year applied AFTER join instead of before

All events data processed through join before filtering

Partition pruning not utilized due to filter placement

Recommendation: Move year filter BEFORE the join operation to reduce join input size and enable partition pruning
############
Query 3: Three-table Join
Status: Needs Optimization

Issues found:

SortMergeJoin between events and tracks creates expensive shuffle (200 partitions)

Genre filter applied AFTER both joins instead of being pushed to artists scan

4 total exchanges creates significant shuffle overhead

Suboptimal join order increases data processed

Recommendation:

Push genre filter to artists scan

Consider broadcasting tracks table if small enough

Reorder joins to filter early
###############
Query 4: Multiple Aggregations
Status: Critical Issue

Issues found:

Complete recomputation of same enriched dataset for each aggregation

Events table scanned twice

Artists table scanned twice

Join executed twice unnecessarily

4 total exchanges across both queries (2 each) doing redundant work

Recommendation: CACHE the enriched DataFrame after the join+filter operation to eliminate all redundant computation
##################
Query 5: Self-join Pattern
Status: Needs Optimization

Issues found:

Events table scanned twice (once for aggregation, once for join)

Aggregation result not cached, forcing recomputation

Could be rewritten to scan events only once

Recommendation:

Cache the popular aggregation result

OR use window functions to compute play counts in a single scan

In [ ]:
## Optimized Query Versions
# Query 2: Events-Artists Join
# PROPOSED: Move filter BEFORE join to reduce data processed
q2_optimized = events \
    .filter(col("year") == 2024) \  # Filter first - reduces join input
    .join(broadcast(artists), "artist_id") \  # Then join with broadcast
    .select("event_id", "name", "genre", "duration_sec")

# EXPLANATION:
# 1. Filter on year is applied BEFORE join, enabling partition pruning
# 2. Only 2024 events participate in the join (reduces join input by ~90%)
# 3. Artists table remains broadcast (optimal for small dimension table)
# 4. Same final result with significantly less data processed

In [ ]:
## Query 3: Three-table Join with Aggregation
# PROPOSITION 1: Optimize with broadcast for tracks (if small enough)
# Check tracks table size first
if spark.table("tracks").count() < 100000:  # Threshold for broadcast
    tracks_for_join = broadcast(tracks)
else:
    tracks_for_join = tracks

q3_optimized = events \
    .filter(col("year") == 2024) \  # Early partition pruning
    .join(tracks_for_join, "track_id") \  # Maybe broadcast, maybe sort-merge
    .join(
        artists.filter(col("genre") == "Pop"),  # Filter pushed to artists scan!
        "artist_id"
    ) \
    .groupBy("name") \
    .agg(
        count("*").alias("play_count"),
        avg("duration_sec").alias("avg_duration")
    )

# PROPOSITION 2: Reorder joins to filter even earlier
q3_optimized_v2 = events \
    .filter(col("year") == 2024) \
    .join(
        artists.filter(col("genre") == "Pop"),  # Join with filtered artists first
        "artist_id"
    ) \
    .join(tracks, "track_id") \  # Then join with tracks
    .groupBy("name") \
    .agg(
        count("*").alias("play_count"),
        avg("duration_sec").alias("avg_duration")
    )

# EXPLANATION:
# 1. Year filter applied immediately (partition pruning)
# 2. Genre filter pushed directly to artists scan (reduces artists rows)
# 3. Option 1 broadcasts tracks if small enough (eliminates SortMergeJoin shuffle)
# 4. Option 2 reorders joins to reduce data earlier
# 5. Both reduce exchange count from 4 to potentially 2-3

In [ ]:
## Query 4: Multiple Aggregations
# PROPOSED: Cache the enriched DataFrame
enriched = events.join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .cache()  # CRITICAL: Cache after join+filter

# Force cache (optional but good practice)
print(f"Enriched dataset cached with {enriched.count()} rows")

print("QUERY 4a: Genre aggregation (using cached data)")
q4a_optimized = enriched \
    .groupBy("genre") \
    .agg(count("*").alias("plays"))
q4a_optimized.explain(mode="formatted")

print("\nQUERY 4b: Device aggregation (using cached data)")
q4b_optimized = enriched \
    .groupBy("device") \
    .agg(avg("duration_sec").alias("avg_dur"))
q4b_optimized.explain(mode="formatted")

# Clean up when done
# enriched.unpersist()

# ALTERNATIVE: If memory is constrained, use MEMORY_AND_DISK persistence
enriched_disk = events.join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .persist(StorageLevel.MEMORY_AND_DISK)

# EXPLANATION:
# 1. .cache() stores the join+filter result in memory after first computation
# 2. Second query reads from cache instead of recomputing
# 3. Eliminates: second events scan, second artists scan, second join
# 4. Reduces exchanges from 4 total to 2 total (just aggregation shuffles)
# 5. Estimated 50% reduction in total execution time

In [ ]:
## Query 5: Self-join Pattern
# PROPOSITION 1: Cache the popular aggregation
popular = events.groupBy("track_id") \
    .agg(count("*").alias("play_count")) \
    .filter(col("play_count") > 10) \
    .cache()  # Cache the small aggregated result

print(f"Popular tracks cached: {popular.count()} tracks")

q5_optimized = events.join(popular, "track_id") \
    .select("event_id", "user_id", "track_id", "play_count")

# PROPOSITION 2: Use window functions (single scan - MOST EFFICIENT)
from pyspark.sql import Window

q5_window_optimized = events \
    .withColumn(
        "play_count",
        count("*").over(Window.partitionBy("track_id"))
    ) \
    .filter(col("play_count") > 10) \
    .select("event_id", "user_id", "track_id", "play_count")

# PROPOSITION 3: CTE approach for clarity (Spark optimizes this)
from pyspark.sql.functions import *

with_popular = events \
    .groupBy("track_id") \
    .agg(count("*").alias("play_count")) \
    .filter(col("play_count") > 10)

q5_cte_optimized = events.alias("e") \
    .join(with_popular.alias("p"), col("e.track_id") == col("p.track_id")) \
    .select("e.event_id", "e.user_id", "e.track_id", "p.play_count")

# EXPLANATION:
# Option 1 (Cache): Popular aggregation computed once, cached, then joined
# - Events scanned twice but aggregation result cached (good if popular is small)
#
# Option 2 (Window - BEST): Single scan of events!
# - Window function computes play_count for each row without separate aggregation
# - No separate aggregation stage, no join
# - Most efficient for large events tables
#
# Option 3 (CTE): Same physical plan as Option 1, just different syntax